In [ ]:
!pip install -qU vllm trl peft datasets pyarrow


### if vllm raise error

In [ ]:
# import os
# import sys
# os.environ["VLLM_USE_V1"] = "0"
# sys.stdout.fileno = lambda: 1
# sys.stderr.fileno = lambda: 2


In [ ]:
import csv
import re
import ast
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
from transformers import AutoTokenizer

MODEL_ID = "google/gemma-3-1b-it"
CHECKPOINT_PATH = "grpo-240"
CSV_PATH = "test_public.csv"
N_CANDIDATES = 16
SYSTEM_PROMPT = "You are a helpful assistant. You first think about the reasoning process in the mind and then provide the user with the answer. Be extremely concise. Focus only on the successful logical path. Avoid any conversational fillers, self-corrections, or trial-and-error markers like 'wait', 'retry', or 'let's try again'."

def verify_equation(equation_str, target, allowed_nums):
    try:
        eq = equation_str.split('=')[0].strip()

        used_nums = [int(n) for n in re.findall(r'\d+', eq)]

        temp_allowed = allowed_nums.copy()
        for n in used_nums:
            if n in temp_allowed:
                temp_allowed.remove(n)
            else:
                return False

        if len(temp_allowed) > 0:
            return False

        result = eval(eq, {"__builtins__": None}, {})

        return abs(result - target) < 1e-6
    except:
        return False

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

llm = LLM(
    model=MODEL_ID,
    enable_lora=True,
    max_lora_rank=128,
    gpu_memory_utilization=0.9,
    max_model_len=3328,
    enforce_eager=True
)

sampling_params = SamplingParams(
    n=N_CANDIDATES,
    temperature=0.7,
    #repetition_penalty=1.1,
    max_tokens=2560,
    stop_token_ids=[tokenizer.eos_token_id]
)

prompts = []
tasks_data = []


with open(CSV_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        nums = ast.literal_eval(row["nums"])
        tgt = int(row["target"])

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
            "role": "user",
            "content": f"Using the numbers {nums}, create an equation that equals {tgt}. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final equation and answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 = 1 </answer>."
            }
        ]

        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append(input_text)
        tasks_data.append({"id": row["id"], "target": tgt, "nums": nums})

outputs = llm.generate(
    prompts,
    sampling_params,
    lora_request=LoRARequest("gemma_adapter", 1, CHECKPOINT_PATH),
    use_tqdm=True
)

output_filename = "gemma_submission.csv"

with open(output_filename, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["id", "equation"])

    for i, output in enumerate(outputs):
        task = tasks_data[i]
        default = "-1"

        for candidate in output.outputs:
            text = candidate.text
            match = re.search(r'<answer>(.*?)</answer>', text, flags=re.DOTALL)

            if match:
                raw_eq = match.group(1).strip()
                if verify_equation(raw_eq, task["target"], task["nums"]):
                    default = raw_eq.split('=')[0].strip()
                    break

        writer.writerow([task["id"], default])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


INFO 04-21 16:37:33 [utils.py:233] non-default args: {'max_model_len': 3328, 'disable_log_stats': True, 'enforce_eager': True, 'enable_lora': True, 'max_lora_rank': 128, 'model': 'google/gemma-3-1b-it'}
WARNING 04-21 16:37:33 [envs.py:1744] Unknown vLLM environment variable detected: VLLM_USE_V1
INFO 04-21 16:37:36 [model.py:549] Resolved architecture: Gemma3ForCausalLM
INFO 04-21 16:37:36 [model.py:1678] Using max model len 3328
INFO 04-21 16:37:36 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-21 16:37:36 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 04-21 16:37:36 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 04-21 16:37:36 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 04-21 16:37:37 [vllm.py:1025] Cudagraph is disabl

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore pid=79261) INFO 04-21 16:37:46 [default_loader.py:384] Loading weights took 0.59 seconds
(EngineCore pid=79261) INFO 04-21 16:37:46 [punica_selector.py:20] Using PunicaWrapperGPU.
(EngineCore pid=79261) INFO 04-21 16:37:47 [gpu_model_runner.py:4820] Model loading took 2.11 GiB memory and 2.832469 seconds
(EngineCore pid=79261) INFO 04-21 16:38:00 [gpu_worker.py:436] Available KV cache memory: 16.7 GiB
(EngineCore pid=79261) WARNING 04-21 16:38:00 [kv_cache_utils.py:1059] Add 2 padding layers, may waste at most 9.09% KV cache memory
(EngineCore pid=79261) INFO 04-21 16:38:00 [kv_cache_utils.py:1319] GPU KV cache size: 625,328 tokens
(EngineCore pid=79261) INFO 04-21 16:38:00 [kv_cache_utils.py:1324] Maximum concurrency for 3,328 tokens per request: 187.13x
(EngineCore pid=79261) INFO 04-21 16:38:01 [core.py:283] init engine (profile, create kv cache, warmup model) took 13.94 seconds
(EngineCore pid=79261) WARNING 04-21 16:38:02 [vllm.py:848] Enforce eager set, disabling torc

Rendering prompts:   0%|          | 0/2000 [00:00<?, ?it/s]

WARNING 04-21 16:38:02 [input_processor.py:149] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts:   0%|          | 0/4000 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

(EngineCore pid=79261) WARNING 04-21 16:38:12 [utils.py:267] Using default LoRA kernel configs
